# v6 · M5: mini-cluster *itemsets* and support-guided local search (deep descriptions, large descriptor space)
Same task design as v5 (Bart-style binary descriptors on the real M5 target, deep true subgroups). New:

- **Itemset support (Apriori-style).** Every mini-cluster (expanded SNN component) yields a *basket*: its most enriched
  conditions. The support of a combination of conditions = the number of mini-clusters whose basket contains **all**
  of them, so a basket {A, B, C} also supports {A}, {B}, {A, B}, … (supersets count toward their subsets).
  `BUS itemsets` evaluates only combinations supported by ≥ MIN_SUPPORT mini-clusters, most-supported first.
- **Support-guided beam local search.** From the best BUS descriptions, keep a small beam (width 5). Moves: drop one
  condition; add or swap in a condition that **co-occurs** (pair support ≥ 1) with a condition already in the
  description, the 20 best-supported ones only. Co-occurrence decides *which* moves are tried; the quality measure
  decides *which* results are kept.
- **Resumable runs.** Tasks run in batches; every batch is appended to the CSV and `progress.txt` is updated, so a
  laptop restart loses at most one batch. Rerunning the notebook skips finished tasks.

In [ ]:
import os
import time
import heapq
import itertools
from collections import Counter

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 110)
N_JOBS = os.cpu_count()
print("CPU cores:", N_JOBS)

In [ ]:
DATASET = "m5"
QUICK = False          # True = tiny grids, only to test the notebook end to end
UNIT = "$"
PERIOD = "day"
PERIODS_PER_YEAR = 365

# ---- Search parameters (Bart §6.3.1) ----
D_MAX = 3                # max conditions in a description
Q = 20                   # result set size
W = 50                   # default beam width
U = 200                  # mini-clusters (pairs) for every BUS variant and both ensembles
POOL_SIZE = 30           # pool ensemble: conditions the BS step may use
W_MC = 5                 # per-mini-cluster ensemble: beam width of each small beam search

# ---- Narrow findings ----
MIN_COV = 0.02
MAX_COV = 0.15
MIN_STORES = 2
MIN_PEERS = 5

# ---- Reference point ----
START_DATE, END_DATE = "2013-01-01", "2016-05-22"   # base window = all of 2013
MIN_UNITS_PER_DAY = 1.0     # keep item x store series selling at least this many units per day in 2013
M5_DIR = "/kaggle/input/m5-forecasting-accuracy" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "M5"
RESOLUTION = "D"

# ---- Experiment 0: real data ----
MEASURES = ["M1", "M2", "M3"]
METHODS = ["BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]
NU = 20 if QUICK else 500                     # permutation runs for the DFD (Bart used 500)

# ---- Experiment 1: planted subgroups ----
PATTERNS = {"level": "M2", "trend": "M2", "timing": "M3"}    # pattern -> the measure that should catch it
SNR = [2, 8] if QUICK else [1, 2, 4, 8]                      # planted effect in units of the null spread sigma (see E1)
LITS = [2] if QUICK else [2, 3]                               # conditions in the planted description
OUTLIERS = [0.0] if QUICK else [0.0, 0.2]                     # share of planted records left unchanged
REPS = 1 if QUICK else 2                                      # different planted descriptions per setting
PLANT_COVERAGE = (0.02, 0.08)
E1_METHODS = ["BS", "BS w=10", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

# ---- Experiment 2: interaction-only subgroups ----
E2_SNR = [8] if QUICK else [4, 8]
E2_REPS = 1 if QUICK else 3
E2_METHODS = ["BS w=5", "BS w=10", "BS w=20", "BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

HIT_JACCARD = 0.8        # a planted subgroup counts as found if a subgroup with Jaccard >= 0.8 was evaluated
SEED = 0
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
RESULTS_DIR = "/kaggle/working/results_v4" if ON_KAGGLE else "results_v4"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ================= v4 experiment settings =================
# pattern -> the quality measure that matches it
PATTERNS_V4 = {"level": "M2", "trend": "M2", "stockout": "M2",      # money patterns (Konijn-style excess)
               "season": "M3", "lag": "M3",                           # shape patterns (Bart phi_cluster)
               "volatility": "M4"}                                    # variance pattern (new M4)
SNR_V4 = [2, 8] if QUICK else [2, 4, 8]           # planted strength in units of the null spread sigma
LAG_STEPS = {2: 1, 4: 2, 8: 4}                    # lag pattern: shift by this many LAG_UNITs instead of snr x sigma
LAG_UNIT = {"day": 7, "week": 1, "month": 1}[PERIOD]
LITS_V4 = [2] if QUICK else [2, 3]
REPS_V4 = 1 if QUICK else 3

# seeds (mini-clusters)
U_SEEDS = 200          # pairs per pair-based seed type
KNN = 10               # neighbours for the kNN / shared-nearest-neighbour graph
SNN_MIN = 5            # edge in the SNN graph if two mutual neighbours share >= SNN_MIN neighbours
Z_COMP_MIN = 3         # smallest non-convex component used as a mini-cluster
MAJORITY = 0.6         # a component describes itself by conditions shared by >= 60% of its members
N_BLOCKS = 36          # time blocks of the denoised representation
LOCAL_TOPK = 10        # Q3: local search starts from BUS's 10 best candidates

SEED_TYPES = ["closest-$", "closest-rel", "closest-blocks", "snn-pairs", "snn-components"]
METHODS_V4 = ["BS w=10", "BS",
              "BUS closest-$", "BUS→BS pool α-$",                       # the old BUS and old pool ensemble
              "BUS closest-rel", "BUS closest-blocks", "BUS snn-pairs", "BUS snn-components",
              "BUS→BS pool blocks", "BUS→BS pool snn",
              "BUS snn-pairs +local", "BUS closest-blocks +local"]
RESULTS_DIR = "/kaggle/working/results_v4" if ON_KAGGLE else "results_v4"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ================= v5 settings =================
K_ATTRS = [100, 400]
LITS_V5 = [3, 4] if QUICK else [3, 4, 5, 6]
SNR_V5 = [8] if QUICK else [8, 16]
OUTLIERS_V5 = [0.2] if QUICK else [0.0, 0.2]
REPS_V5 = 1 if QUICK else 15
MIN_COV_V5 = 0.01
TOP_ENRICHED = 8            # conditions per mini-cluster used to build descriptions
METHODS_V5 = ["BS w=10", "BS w=50", "BUS snn-comp", "BUS snn-comp +local", "BUS→BS pool snn-comp"]
RESULTS_DIR = "results_v5"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ================= v6 settings (override v5) =================
MIN_SUPPORT = 2            # an itemset must be contained in >= 2 mini-cluster baskets
GUIDED_WIDTH = 5           # beam width of the guided local search
GUIDED_ADD_TOP = 20        # best-supported co-occurring conditions tried per step
METHODS_V5 = ["BS w=10", "BS w=50", "BUS snn-comp", "BUS snn-comp +local", "BUS→BS pool snn-comp",
              "BUS itemsets", "BUS itemsets +guided beam"]
RESULTS_DIR = "results_v6"
os.makedirs(RESULTS_DIR, exist_ok=True)
BATCH = 2 * N_JOBS

In [ ]:
calendar = pd.read_csv(f"{M5_DIR}/calendar.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
sales_wide = pd.read_csv(f"{M5_DIR}/sales_train_evaluation.csv")
prices = pd.read_csv(f"{M5_DIR}/sell_prices.csv")

day_cols = [c for c in sales_wide.columns if c.startswith("d_")]
all_dates = calendar["date"].iloc[[int(c[2:]) - 1 for c in day_cols]].reset_index(drop=True)
in_period = (all_dates >= START_DATE) & (all_dates <= END_DATE)
dates = pd.DatetimeIndex(all_dates[in_period])
units = sales_wide[day_cols].to_numpy(dtype=np.float32)[:, in_period.to_numpy()]
BASE_LEN = int((dates < "2014-01-01").sum())

keep = units[:, :BASE_LEN].mean(axis=1) >= MIN_UNITS_PER_DAY
sales_wide, units = sales_wide[keep].reset_index(drop=True), units[keep]

# Dollars = units x the weekly shelf price of that item in that store (filled across weeks without a price).
week_of_day = calendar.set_index("date").loc[dates, "wm_yr_wk"].to_numpy()
weeks = np.unique(week_of_day)
price_wide = (prices.pivot_table(index=["store_id", "item_id"], columns="wm_yr_wk", values="sell_price")
                    .reindex(columns=weeks))
key = pd.MultiIndex.from_frame(sales_wide[["store_id", "item_id"]])
price_wide = price_wide.reindex(key).ffill(axis=1).bfill(axis=1)
price_daily = price_wide.to_numpy()[:, np.searchsorted(weeks, week_of_day)]
S = units.astype(float) * price_daily
if RESOLUTION == "W":                                    # same days, summed into 7-day blocks
    n_weeks = S.shape[1] // 7
    S = S[:, : n_weeks * 7].reshape(len(S), n_weeks, 7).sum(axis=2)
    dates = dates[: n_weeks * 7: 7]
    BASE_LEN = int((dates < "2014-01-01").sum())
MIN_BASE_LEVEL = 0.0       # the size filter was applied on units above

records = sales_wide[["store_id", "item_id", "dept_id", "cat_id", "state_id"]].rename(
    columns={"store_id": "store", "item_id": "item"})

# Item-level descriptors (same in every store): price level, price volatility, launch week.
item_prices = prices.groupby("item_id")["sell_price"]
first_week = prices.groupby("item_id")["wm_yr_wk"].min()
records["item_price"] = records["item"].map(item_prices.mean())
records["item_price_volatility"] = records["item"].map(item_prices.std() / item_prices.mean())
records["item_launch"] = records["item"].map(first_week.rank())

# Record-level descriptors (this store vs. the same item elsewhere): price level, price trend, discount frequency.
p = price_wide.to_numpy()
t_w = np.arange(p.shape[1], dtype=float)
t_w -= t_w.mean()
records["price_level"] = p.mean(axis=1)
records["price_trend"] = (p - p.mean(axis=1, keepdims=True)) @ t_w / (t_w ** 2).sum() * 52 / p.mean(axis=1)
records["discount_freq"] = (p < 0.95 * p.max(axis=1, keepdims=True)).mean(axis=1)

categorical_desc = [("state_id", "store"), ("cat_id", "item"), ("dept_id", "item")]
numeric_desc = [("item_price", "dept_id", "item"), ("item_price_volatility", None, "item"), ("item_launch", None, "item"),
                ("price_level", "item", "record"), ("price_trend", "item", "record"), ("discount_freq", "item", "record")]
del units, price_daily, sales_wide
print(S.shape, records.shape)

In [ ]:
level = S[:, :BASE_LEN].mean(axis=1)
item = records["item"].astype(str).to_numpy()

ok = level >= MIN_BASE_LEVEL
ok &= pd.Series(ok).groupby(item).transform("sum").to_numpy() >= MIN_PEERS
records = records[ok].reset_index(drop=True)
S, level, item = S[ok], level[ok], item[ok]
N, T = S.shape

group_sales = pd.DataFrame(S).groupby(item).transform("sum").to_numpy()
group_level = pd.Series(level).groupby(item).transform("sum").to_numpy()
peer_index = (group_sales - S) / (group_level - level)[:, None]
expected = level[:, None] * peer_index
excess = S - expected
rel_dev = np.clip(excess / level[:, None], -3, 3)
store_codes = pd.factorize(records["store"])[0]
del group_sales, peer_index

print(f"{DATASET}: N = {N} records, T = {T} {PERIOD}s, {records['store'].nunique()} stores, "
      f"{records['item'].nunique()} items, base window = {BASE_LEN} {PERIOD}s")
print(f"Base-window excess per record (0 by construction): max |.| = {np.abs(excess[:, :BASE_LEN].sum(axis=1)).max():.6f}")

In [ ]:
LABELS = ["very_low", "low", "medium", "high", "very_high"]


def z_bins(z):
    return pd.cut(z, bins=[-np.inf, -1.5, -0.5, 0.5, 1.5, np.inf], labels=LABELS).astype(str)


descriptor_cols, desc_level = [], {}
for column, level_ in categorical_desc:
    records[column] = records[column].astype(str)
    descriptor_cols.append(column)
    desc_level[column] = level_
for column, zgroup, level_ in numeric_desc:
    values = records[column].astype(float)
    if zgroup is None:
        z = (values - values.mean()) / values.std()
        name = f"{column}_cat"
    else:
        grouped = values.groupby(records[zgroup])
        z = (values - grouped.transform("mean")) / grouped.transform("std").replace(0, np.nan)
        name = f"{column}_vs_{zgroup}_cat"
    records[name] = z_bins(z.fillna(0))
    descriptor_cols.append(name)
    desc_level[name] = level_

A = records[descriptor_cols].astype(str).to_numpy(dtype=object)


def describe(desc):
    return " AND ".join(f"{descriptor_cols[j]} = {v}" for j, v in desc)


print(f"{len(descriptor_cols)} descriptors, {sum(records[c].nunique() for c in descriptor_cols)} conditions")
pd.DataFrame({"level": desc_level, "distinct values": records[descriptor_cols].nunique()})

In [ ]:
class QualityBase:
    def __init__(self, N, store_codes, min_cov=None, max_cov=None, min_stores=None):
        min_cov = MIN_COV if min_cov is None else min_cov
        max_cov = MAX_COV if max_cov is None else max_cov
        self.N = N
        self.store_codes = store_codes
        self.n_store_codes = int(store_codes.max()) + 1
        self.min_n = max(2, int(np.ceil(min_cov * N)))
        self.max_n = int(np.floor(max_cov * N))
        self.min_stores = MIN_STORES if min_stores is None else min_stores
        self.cache = {}
        self.track = None        # planted mask, if the evaluations should be traced
        self.trace = []          # (time, quality, jaccard with planted, valid)

    def n_stores(self, mask):
        return np.count_nonzero(np.bincount(self.store_codes[mask], minlength=self.n_store_codes))

    def expandable(self, mask):
        return mask.sum() >= self.min_n and self.n_stores(mask) >= self.min_stores

    def satisfies_constraints(self, mask):
        n = mask.sum()
        return n <= self.max_n and n < self.N and self.expandable(mask)

    def optimistic_estimate(self, mask):
        return None

    def __call__(self, mask):
        key = mask.tobytes()
        if key not in self.cache:
            self.cache[key] = self.compute(mask)
            if self.track is not None:
                inter = np.count_nonzero(mask & self.track)
                union = np.count_nonzero(mask | self.track)
                self.trace.append((time.time(), self.cache[key], inter / union, self.satisfies_constraints(mask)))
        return self.cache[key]


class MonetaryQuality(QualityBase):
    def __init__(self, excess, level, store_codes, relative=True, **constraints):
        super().__init__(len(level), store_codes, **constraints)
        self.relative = relative
        self.x = excess.sum(axis=1, dtype=np.float64)
        self.level = level
        self.L_T = level.sum()
        self.y = self.x - level * self.x.sum() / self.L_T

    def signed(self, mask):
        if not self.relative:
            return self.x[mask].sum()
        return self.y[mask].sum() * self.L_T / (self.L_T - self.level[mask].sum())

    def compute(self, mask):
        return abs(self.signed(mask))

    def optimistic_estimate(self, mask):
        v = self.y[mask] if self.relative else self.x[mask]
        bound = max(v[v > 0].sum(), -v[v < 0].sum())
        if self.relative:
            bound *= self.L_T / (self.L_T - self.level[mask].sum())
        return bound


class ClusterQuality(QualityBase):
    def __init__(self, D, store_codes, **constraints):
        super().__init__(len(D), store_codes, **constraints)
        self.D = D
        self.rowsum = D.sum(axis=1, dtype=np.float64)

    def components(self, mask):
        idx = np.flatnonzero(mask)
        n, n_c = len(idx), self.N - len(idx)
        within = self.D[np.ix_(idx, idx)].sum(dtype=np.float64)
        between = self.rowsum[idx].sum() - within
        return between / (n * n_c), within / (n * (n - 1))

    def compute(self, mask):
        alpha, beta = self.components(mask)
        return alpha / (beta + 1)


def _cityblock_block(Xb, X):
    return cdist(Xb, X, metric="cityblock")


def pairwise_cityblock(X, block=512):
    """Full N x N city-block distance matrix, row blocks computed in parallel."""
    N_ = len(X)
    dtype = np.float32 if N_ > 6000 else np.float64
    starts = list(range(0, N_, block))
    parts = Parallel(n_jobs=N_JOBS)(delayed(_cityblock_block)(X[s:s + block], X) for s in starts)
    D = np.empty((N_, N_), dtype=dtype)
    for s, part in zip(starts, parts):
        D[s:s + len(part)] = part
    return D


def update_rows(D, X, rows):
    """Distances after the target rows `rows` changed: only those rows / columns are recomputed."""
    D = D.copy()
    if len(rows):
        new = cdist(X[rows], X, metric="cityblock").astype(D.dtype)
        D[rows, :] = new
        D[:, rows] = new.T
    return D


start = time.time()
D_money = pairwise_cityblock(excess)
D_shape = pairwise_cityblock(rel_dev)
DISTANCE_TIME = time.time() - start
DISTANCE_FOR = {"M1": D_money, "M2": D_money, "M3": D_shape}
print(f"Distance matrices ({N} x {N}, T = {T}, {D_money.dtype}) built in {DISTANCE_TIME:.1f}s")


def make_quality(measure, exc=None, D=None):
    exc = excess if exc is None else exc
    if measure == "M1":
        return MonetaryQuality(exc, level, store_codes, relative=False)
    if measure == "M2":
        return MonetaryQuality(exc, level, store_codes, relative=True)
    return ClusterQuality(D_shape if D is None else D, store_codes)


_phi = make_quality("M2")
_mask = np.zeros(N, dtype=bool)
_mask[: N // 10] = True
_direct = _phi.x[_mask].sum() - level[_mask].sum() * (_phi.x.sum() - _phi.x[_mask].sum()) / (level.sum() - level[_mask].sum())
assert np.isclose(_phi.signed(_mask), _direct)

In [ ]:
def mask_of(A, desc):
    mask = np.ones(len(A), dtype=bool)
    for j, v in desc:
        mask &= A[:, j] == v
    return mask


class ResultSet:
    def __init__(self, q):
        self.q = q
        self.items = {}

    def insert(self, quality, desc, mask):
        key = mask.tobytes()
        old = self.items.get(key)
        if old is not None and len(old[1]) <= len(desc):
            return
        self.items[key] = (quality, desc, mask)
        if len(self.items) > self.q:
            del self.items[min(self.items, key=lambda k: self.items[k][0])]

    def full(self):
        return len(self.items) >= self.q

    def threshold(self):
        return min(v[0] for v in self.items.values()) if self.full() else -np.inf

    def sorted(self):
        return sorted(self.items.values(), key=lambda x: -x[0])


def oe_below_threshold(phi, mask, result_set):
    if not result_set.full():
        return False
    oe = phi.optimistic_estimate(mask)
    return oe is not None and oe < result_set.threshold()


def refine(seed, seed_mask, A, pool=None):
    used = {j for j, _ in seed}
    if pool is None:
        options = ((j, v) for j in range(A.shape[1]) if j not in used for v in pd.unique(A[seed_mask, j]))
    else:
        options = ((j, v) for j, v in pool if j not in used and (A[seed_mask, j] == v).any())
    for condition in options:
        yield tuple(sorted(seed + (condition,), key=lambda c: c[0]))


def _beam(A, phi, pool, w, d, prune, result_set, stats, evaluated_keys, mask_cache):
    def get_mask(desc):
        if desc not in mask_cache:
            mask_cache[desc] = mask_of(A, desc)
        return mask_cache[desc]

    candidate_queue = [()]
    seen, tie = set(), itertools.count()
    for level_ in range(1, d + 1):
        beam = []
        for seed in candidate_queue:
            seed_mask = get_mask(seed)
            if prune and seed and oe_below_threshold(phi, seed_mask, result_set):
                stats["pruned_oe_seed"] += 1
                continue
            for desc in refine(seed, seed_mask, A, pool):
                if desc in seen:
                    continue
                seen.add(desc)
                stats["generated"] += 1
                mask = get_mask(desc)
                if not phi.expandable(mask):
                    stats["skipped_coverage"] += 1
                    continue
                if prune and seed and mask.sum() == seed_mask.sum():
                    stats["pruned_redundant"] += 1
                    continue
                quality = phi(mask)
                stats["visits"] += 1
                evaluated_keys.add(desc)
                if phi.satisfies_constraints(mask):
                    result_set.insert(quality, desc, mask)
                if prune and oe_below_threshold(phi, mask, result_set):
                    stats["pruned_oe"] += 1
                    continue
                item_ = (quality, next(tie), desc)
                if len(beam) < w:
                    heapq.heappush(beam, item_)
                else:
                    heapq.heappushpop(beam, item_)
        candidate_queue = [desc for _, _, desc in sorted(beam, reverse=True)]


def beam_search(A, phi, w=W, d=D_MAX, q=Q, prune=False, pool=None):
    stats, result_set, keys = Counter(), ResultSet(q), set()
    _beam(A, phi, pool, w, d, prune, result_set, stats, keys, {})
    stats["evaluated"] = len(keys)
    return result_set.sorted(), stats


def top_pairs(D, u=U, kind="alpha", block=512):
    """Block-wise pair selection over the lower triangle.
    kind="closest": the u smallest distances (plain BUS, Alg. 2).
    kind="alpha":   the u largest alpha_ij / (D_ij + 1) (alpha-matrix, §4.3.2)."""
    N_ = len(D)
    rowsum = D.sum(axis=1, dtype=np.float64)
    scores, rows_, cols_ = [], [], []
    for s in range(0, N_, block):
        Db = D[s:s + block].astype(np.float64)
        i = np.arange(s, s + len(Db))[:, None]
        j = np.arange(N_)[None, :]
        if kind == "closest":
            score = -Db
        else:
            alpha = (rowsum[i] + rowsum[j] - 2 * Db) / (2 * (N_ - 2))
            score = alpha / (Db + 1)
        score = np.where(j < i, score, -np.inf).ravel()
        k = min(u, int(np.isfinite(score).sum()))
        if k == 0:
            continue
        best = np.argpartition(-score, k - 1)[:k]
        scores.append(score[best])
        rows_.append(s + best // N_)
        cols_.append(best % N_)
    scores, rows_, cols_ = np.concatenate(scores), np.concatenate(rows_), np.concatenate(cols_)
    order = np.argsort(-scores)[:u]
    return [(int(rows_[o]), int(cols_[o])) for o in order]


def shared_conditions(mc, A):
    first = mc[0]
    return [(j, A[first, j]) for j in range(A.shape[1]) if all(A[i, j] == A[first, j] for i in mc)]


def evaluate_candidates(A, phi, candidates, q=Q, prune=False):
    stats = Counter(generated=len(candidates))
    result_set = ResultSet(q)
    dead = set()
    for desc in sorted(candidates, key=len):
        if prune and len(desc) > 1 and any(
            sub in dead for size in range(1, len(desc)) for sub in itertools.combinations(desc, size)
        ):
            stats["pruned_superset"] += 1
            continue
        mask = mask_of(A, desc)
        if not phi.expandable(mask):
            stats["skipped_coverage"] += 1
            dead.add(desc)
            continue
        quality = phi(mask)
        stats["evaluated"] += 1
        stats["visits"] += 1
        if phi.satisfies_constraints(mask):
            result_set.insert(quality, desc, mask)
        if prune and oe_below_threshold(phi, mask, result_set):
            dead.add(desc)
    return result_set.sorted(), stats


def bus(A, phi, pairs, d=D_MAX, q=Q, prune=False):
    candidates = set()
    for mc in pairs:
        conditions = shared_conditions(mc, A)
        for size in range(1, d + 1):
            candidates.update(itertools.combinations(conditions, size))
    top, stats = evaluate_candidates(A, phi, candidates, q, prune)
    stats["mini_clusters"] = len(pairs)
    return top, stats


def ensemble_pool(A, phi, pairs, pool_size=POOL_SIZE, prune=False):
    votes = Counter(c for mc in pairs for c in shared_conditions(mc, A))
    pool = [c for c, _ in votes.most_common(pool_size)]
    top, stats = beam_search(A, phi, prune=prune, pool=pool)
    stats["mini_clusters"] = len(pairs)
    stats["pool"] = len(pool)
    return top, stats, pool


def ensemble_per_mc(A, phi, pairs, w=W_MC, d=D_MAX, q=Q, prune=False):
    stats, result_set, keys, mask_cache = Counter(), ResultSet(q), set(), {}
    spaces = set()
    for mc in pairs:
        space = tuple(shared_conditions(mc, A))
        if not space:
            stats["mc_empty"] += 1
            continue
        if space in spaces:
            stats["mc_duplicate_space"] += 1
            continue
        spaces.add(space)
        _beam(A, phi, list(space), w, d, prune, result_set, stats, keys, mask_cache)
    stats["evaluated"] = len(keys)
    stats["mini_clusters"] = len(pairs)
    stats["mc_spaces"] = len(spaces)
    union = sorted({c for space in spaces for c in space}, key=lambda c: (c[0], str(c[1])))
    return result_set.sorted(), stats, union


def run_method(method, A, phi, D, prune, pairs=None):
    """method: "BS", "BS w=<k>", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC".
    Runtime includes mini-cluster selection unless precomputed pairs are passed."""
    start = time.time()
    pool = None
    if method.startswith("BS"):
        width = int(method.split("w=")[1]) if "w=" in method else W
        top, stats = beam_search(A, phi, w=width, prune=prune)
    else:
        if pairs is None:
            pairs = top_pairs(D, kind="closest" if method == "BUS" else "alpha")
        stats_pair = time.time() - start
        if method in ("BUS", "BUS-α"):
            top, stats = bus(A, phi, pairs, prune=prune)
        elif method == "BUS→BS pool":
            top, stats, pool = ensemble_pool(A, phi, pairs, prune=prune)
        else:
            top, stats, pool = ensemble_per_mc(A, phi, pairs, prune=prune)
        stats["pair_time"] = stats_pair
    stats["runtime"] = time.time() - start
    return top, stats, pool


def top_keys(top):
    return {m.tobytes() for _, _, m in top}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 1.0

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

expected = S - excess
POST = np.arange(T) >= BASE_LEN
SIN = np.sin(2 * np.pi * np.arange(T) / PERIODS_PER_YEAR)


class VolatilityQuality(QualityBase):
    """M4: how much more (or less) the subgroup's deviation from its peers fluctuates after the base window.
    v_i = log(var post / var base) of r_i(t); quality = |mean v in S - mean v in complement| * sqrt(n n_c / N)."""

    def __init__(self, rel, store_codes, **constraints):
        super().__init__(len(rel), store_codes, **constraints)
        self.v = np.log((rel[:, BASE_LEN:].var(axis=1) + 1e-6) / (rel[:, :BASE_LEN].var(axis=1) + 1e-6))
        self.total = self.v.sum()

    def compute(self, mask):
        n = mask.sum()
        s = self.v[mask].sum()
        return abs(s / n - (self.total - s) / (self.N - n)) * np.sqrt(n * (self.N - n) / self.N)


def block_means(rel):
    post = rel[:, BASE_LEN:]
    k = max(1, post.shape[1] // N_BLOCKS)
    m = (post.shape[1] // k) * k
    return post[:, :m].reshape(len(rel), -1, k).mean(axis=2)


def apply_pattern(kind, rows, strength, rng, excess, expected, level):
    """Returns the planted excess. strength: share of normal sales (level/trend/season), share of post-base
    periods zeroed (stockout), extra fluctuation (volatility) or periods of lag (lag)."""
    exc = excess.copy()
    if kind == "level":
        exc[rows] += strength * POST[None, :] * level[rows, None]
    elif kind == "trend":
        ramp = np.clip((np.arange(T) - BASE_LEN + 1) / (T - BASE_LEN), 0, None) * POST
        exc[rows] += 2 * strength * ramp[None, :] * level[rows, None]
    elif kind == "season":
        exc[rows] += strength * SIN[None, :] * level[rows, None]
    elif kind == "stockout":
        actual = exc[rows] + expected[rows]
        lost = (rng.random(actual.shape) < strength) & POST[None, :]
        exc[rows] = np.where(lost, 0.0, actual) - expected[rows]
    elif kind == "lag":
        actual = exc[rows] + expected[rows]
        k = int(strength)
        shifted = np.concatenate([np.repeat(actual[:, :1], k, axis=1), actual[:, :-k]], axis=1)
        exc[rows] = shifted - expected[rows]
    elif kind == "volatility":
        post = exc[rows][:, POST]
        mean = post.mean(axis=1, keepdims=True)
        block = exc[rows]
        block[:, POST] = mean + (1 + strength) * (post - mean)
        exc[rows] = block
    return exc


# ---------------- seeds (mini-clusters)
def knn_lists(D, k=KNN, block=1024):
    out = np.empty((len(D), k), dtype=np.int64)
    for s in range(0, len(D), block):
        Db = np.array(D[s:s + block], dtype=np.float32)
        Db[np.arange(len(Db)), np.arange(s, s + len(Db))] = np.inf
        part = np.argpartition(Db, k, axis=1)[:, :k]
        out[s:s + len(Db)] = part
    return out


def snn_edges(D):
    knn = knn_lists(D)
    sets = [set(r) for r in knn.tolist()]
    edges = []
    for i, row in enumerate(knn.tolist()):
        for j in row:
            if j > i and i in sets[j]:
                edges.append((len(sets[i] & sets[j]), float(D[i, j]), i, j))
    return edges


def seeds_snn_pairs(D, u=U_SEEDS):
    edges = sorted(snn_edges(D), key=lambda e: (-e[0], e[1]))
    return [(i, j) for _, _, i, j in edges[:u]]


def seeds_snn_components(D, u=U_SEEDS):
    edges = [e for e in snn_edges(D) if e[0] >= SNN_MIN]
    if not edges:
        return []
    i = np.array([e[2] for e in edges])
    j = np.array([e[3] for e in edges])
    graph = coo_matrix((np.ones(len(i)), (i, j)), shape=(len(D), len(D)))
    _, labels = connected_components(graph, directed=False)
    comps = pd.Series(np.arange(len(D))).groupby(labels).apply(list)
    max_size = max(50, int(0.05 * len(D)))
    comps = [tuple(c) for c in comps if Z_COMP_MIN <= len(c) <= max_size]
    return sorted(comps, key=len, reverse=True)[:u]


def conditions_of(mc, A_, rule):
    if rule == "all":
        return shared_conditions(mc, A_)
    sub = A_[list(mc)]
    conds = []
    for j in range(A_.shape[1]):
        values, counts = np.unique(sub[:, j].astype(str), return_counts=True)
        best = counts.argmax()
        if counts[best] >= MAJORITY * len(mc):
            conds.append((j, values[best]))
    return conds


def candidates_of(mcs, A_, rule):
    cands = set()
    for mc in mcs:
        conds = conditions_of(mc, A_, rule)
        for size in range(1, D_MAX + 1):
            cands.update(itertools.combinations(conds, size))
    return cands


def pool_of(mcs, A_, rule, pool_size=POOL_SIZE):
    votes = Counter(c for mc in mcs for c in conditions_of(mc, A_, rule))
    return [c for c, _ in votes.most_common(pool_size)]


# ---------------- Q3: local search from BUS candidates
def local_search(A_, phi, starts, result_set, stats, max_rounds=10):
    """Hill climbing in the lattice from the best BUS candidates: drop one condition, add one condition
    (values present in the subgroup), or swap the value of one condition; move while quality improves."""
    frontier = [desc for _, desc, _ in starts[:LOCAL_TOPK]]
    seen = set(frontier)
    for _ in range(max_rounds):
        nxt = []
        for desc in frontier:
            mask = mask_of(A_, desc)
            best_q, best_d = phi(mask) if phi.expandable(mask) else -np.inf, None
            neighbours = []
            if len(desc) > 1:
                neighbours += [tuple(c for c in desc if c != drop) for drop in desc]
            used = {j for j, _ in desc}
            if len(desc) < D_MAX:
                for j in range(A_.shape[1]):
                    if j not in used:
                        neighbours += [tuple(sorted(desc + ((j, v),), key=lambda c: c[0])) for v in pd.unique(A_[mask, j])]
            for c in desc:
                rest = tuple(x for x in desc if x != c)
                for v in pd.unique(A_[mask_of(A_, rest), c[0]]):
                    if v != c[1]:
                        neighbours.append(tuple(sorted(rest + ((c[0], v),), key=lambda x: x[0])))
            for nb in neighbours:
                if nb in seen:
                    continue
                seen.add(nb)
                m = mask_of(A_, nb)
                if not phi.expandable(m):
                    continue
                q = phi(m)
                stats["local_evaluated"] += 1
                if phi.satisfies_constraints(m):
                    result_set.insert(q, nb, m)
                if q > best_q:
                    best_q, best_d = q, nb
            if best_d is not None:
                nxt.append(best_d)
        frontier = nxt
        if not frontier:
            break
    return result_set.sorted()


def _rank_of_planted(entries):
    planted_q = [q for _, q, jac, valid in entries if valid and jac >= HIT_JACCARD]
    if not planted_q:
        return np.nan
    best = max(planted_q)
    return 1 + sum(1 for _, q, jac, valid in entries if valid and jac < HIT_JACCARD and q > best)


BUDGETS = [25, 50, 100, 200, 400, 800, 1600, 3200]
SECONDS = [0.1, 0.25, 0.5, 1, 2, 5, 10, 30]

In [ ]:
POST = np.arange(T) >= BASE_LEN


def binary_descriptors(rng, k):
    """Bart's generator: every record gets k attributes ~ Bin(1, 0.5)."""
    return (rng.random((N, k)) < 0.5).astype(np.int8)


def snn_components(D, max_size):
    """Non-convex mini-clusters: connected components of the shared-nearest-neighbour graph, each expanded with its
    members' nearest neighbours (so a ~5-record chain becomes a ~50-record neighbourhood that follows the group)."""
    knn = knn_lists(D)
    edges = [e for e in snn_edges(D) if e[0] >= SNN_MIN]
    if not edges:
        return []
    i = np.array([e[2] for e in edges])
    j = np.array([e[3] for e in edges])
    graph = coo_matrix((np.ones(len(i)), (i, j)), shape=(len(D), len(D)))
    _, labels = connected_components(graph, directed=False)
    comps = pd.Series(np.arange(len(D))).groupby(labels).apply(list)
    comps = [c for c in comps if len(c) >= Z_COMP_MIN]
    expanded = []
    for c in comps:
        members = np.unique(np.concatenate([np.asarray(c), knn[list(c)].ravel()]))
        if len(members) <= max_size:
            expanded.append(tuple(members.tolist()))
    return sorted(set(expanded), key=len, reverse=True)[:U_SEEDS]


def enriched_conditions(mc, A_, overall, z_min=2.0):
    """Conditions shared by >= MAJORITY of the mini-cluster, ranked by a binomial enrichment z-score
    (share in cluster - share overall) * sqrt(n) / sqrt(p (1 - p)). In a large binary space many conditions are shared
    by chance in a small group; the z-score only trusts enrichment that is backed by enough records."""
    sub = A_[list(mc)]
    n = len(mc)
    share1 = sub.mean(axis=0)
    conds = []
    for j in range(A_.shape[1]):
        for v, s, base in ((1, share1[j], overall[j]), (0, 1 - share1[j], 1 - overall[j])):
            if s >= MAJORITY and 0 < base < 1:
                z = (s - base) * np.sqrt(n) / np.sqrt(base * (1 - base))
                if z >= z_min:
                    conds.append((z, (j, v)))
    conds.sort(reverse=True)
    return [c for _, c in conds[:TOP_ENRICHED]]


def local_search_d(A_, phi, starts, result_set, stats, d, topk=LOCAL_TOPK, max_rounds=10):
    frontier = [desc for _, desc, _ in starts[:topk]]
    seen = set(frontier)
    for _ in range(max_rounds):
        nxt = []
        for desc in frontier:
            mask = mask_of(A_, desc)
            best_q, best_d = (phi(mask) if phi.expandable(mask) else -np.inf), None
            neighbours = [tuple(c for c in desc if c != drop) for drop in desc] if len(desc) > 1 else []
            used = {j for j, _ in desc}
            if len(desc) < d:
                for j in range(A_.shape[1]):
                    if j not in used:
                        neighbours += [tuple(sorted(desc + ((j, v),))) for v in np.unique(A_[mask, j])]
            for c in desc:
                rest = tuple(x for x in desc if x != c)
                for v in np.unique(A_[mask_of(A_, rest), c[0]]):
                    if v != c[1]:
                        neighbours.append(tuple(sorted(rest + ((c[0], int(v)),))))
            for nb in neighbours:
                if nb in seen or not nb:
                    continue
                seen.add(nb)
                m = mask_of(A_, nb)
                if not phi.expandable(m):
                    continue
                q = phi(m)
                stats["local_evaluated"] += 1
                if phi.satisfies_constraints(m):
                    result_set.insert(q, nb, m)
                if q > best_q:
                    best_q, best_d = q, nb
            if best_d is not None:
                nxt.append(best_d)
        frontier = nxt
        if not frontier:
            break
    return result_set.sorted()


def _deep_job(spec, excess, level, D_blocks, store_codes, methods):
    rng = np.random.default_rng(spec["seed"])
    k, lits, d = spec["k"], spec["lits"], spec["lits"]
    A_ = binary_descriptors(rng, k)
    attrs = rng.choice(k, lits, replace=False)
    planted_desc = tuple(sorted((int(j), 0) for j in attrs))
    planted = mask_of(A_, planted_desc)
    rows = np.flatnonzero(planted)
    keep = rows[rng.random(len(rows)) >= spec["outliers"]]
    exc = excess.copy()
    exc[keep] += spec["delta"] * POST[None, :] * level[keep, None]
    rel = np.clip(exc / level[:, None], -3, 3)

    t = time.time()
    Db = update_rows(D_blocks, block_means(rel), keep)
    comps = snn_components(Db, max_size=int(MAX_COV * len(level)))
    overall = A_.mean(axis=0)
    comp_conds = [enriched_conditions(mc, A_, overall) for mc in comps]
    seed_time = time.time() - t
    cands = set()
    for conds in comp_conds:
        for size in range(1, min(d, len(conds)) + 1):
            cands.update(tuple(sorted(c)) for c in itertools.combinations(conds, size))
    planted_in = [planted[list(mc)].mean() for mc in comps]
    seed_row = {"components": len(comps), "mean_comp_size": np.mean([len(c) for c in comps]) if comps else 0,
                "best_comp_purity": max(planted_in, default=0.0),
                "comps_mostly_planted": int(sum(p >= 0.5 for p in planted_in)),
                "planted_conds_in_top": max((len(set(cc) & set(planted_desc)) for cc in comp_conds), default=0),
                "candidates": len(cands), "seed_time_s": seed_time}

    out = []
    for method in methods:
        phi = MonetaryQuality(exc, level, store_codes, relative=True, min_cov=MIN_COV_V5)
        phi.track, phi.trace = planted, []
        stats = Counter()
        t0 = time.time()
        offset = 0.0
        if method.startswith("BS"):
            top, stats = beam_search(A_, phi, w=int(method.split("w=")[1]), d=d, prune=True)
        elif method.startswith("BUS→BS pool"):
            offset = seed_time
            votes = Counter(c for conds in comp_conds for c in conds)
            pool = [c for c, _ in votes.most_common(POOL_SIZE)]
            top, stats = beam_search(A_, phi, w=10, d=d, prune=True, pool=pool)
        else:
            offset = seed_time
            top, stats = evaluate_candidates(A_, phi, cands, prune=True)
            if method.endswith("+local"):
                rs = ResultSet(Q)
                for q_, d_, m_ in top:
                    rs.insert(q_, d_, m_)
                top = local_search_d(A_, phi, top, rs, stats, d)
        trace = [(tt - t0 + offset, q_, jac, valid) for tt, q_, jac, valid in phi.trace]
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = {k_: v for k_, v in spec.items()}
        row.update({"method": method, "planted_n": int(planted.sum()), "conditions": 2 * k,
                    "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
                    "rank": ranks[0] if ranks else np.nan, "evaluated": len(trace),
                    "runtime_s": time.time() - t0 + offset, "local_evaluated": stats.get("local_evaluated", 0),
                    **seed_row})
        for b in BUDGETS + [6400, 12800]:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s in SECONDS:
            row[f"rank_at_{s}s"] = _rank_of_planted([e for e in trace if e[0] <= s])
        out.append(row)
    return out

In [ ]:
def itemset_support(baskets, d):
    """Support of every combination (up to d conditions) = number of baskets containing all of it.
    Baskets are small (<= TOP_ENRICHED conditions), so counting all sub-combinations per basket is exact and cheap;
    this is what Apriori computes level by level for large baskets."""
    support = Counter()
    for basket in baskets:
        items = sorted(set(basket))
        for size in range(1, min(d, len(items)) + 1):
            support.update(itertools.combinations(items, size))
    return support


def guided_beam(A_, phi, starts, result_set, stats, d, support, width=GUIDED_WIDTH, max_rounds=10):
    """Small-beam local search whose add / swap moves are restricted to conditions that co-occur (pair support >= 1)
    with a condition already in the description, best-supported first."""
    pair = {k: v for k, v in support.items() if len(k) == 2}
    partners = {}
    for (a, b), s in pair.items():
        partners.setdefault(a, Counter())[b] += s
        partners.setdefault(b, Counter())[a] += s

    def suggestions(desc):
        score = Counter()
        for c in desc:
            score.update(partners.get(c, {}))
        used = {j for j, _ in desc}
        return [c for c, _ in score.most_common() if c[0] not in used][:GUIDED_ADD_TOP]

    def quality(desc):
        m = mask_of(A_, desc)
        if not phi.expandable(m):
            return None
        q = phi(m)
        stats["local_evaluated"] += 1
        if phi.satisfies_constraints(m):
            result_set.insert(q, desc, m)
        return q

    beam = [(q, desc) for q, desc, _ in starts[:width]]
    seen = {desc for _, desc in beam}
    for _ in range(max_rounds):
        cands = []
        for _, desc in beam:
            neighbours = [tuple(c for c in desc if c != drop) for drop in desc] if len(desc) > 1 else []
            for c in suggestions(desc):
                if len(desc) < d:
                    neighbours.append(tuple(sorted(desc + (c,))))
                for old in desc:
                    neighbours.append(tuple(sorted(tuple(x for x in desc if x != old) + (c,))))
            for nb in neighbours:
                if nb and nb not in seen:
                    seen.add(nb)
                    q = quality(nb)
                    if q is not None:
                        cands.append((q, nb))
        new_beam = sorted(beam + cands, key=lambda x: -x[0])[:width]
        if {x for _, x in new_beam} == {x for _, x in beam}:
            break
        beam = new_beam
    return result_set.sorted()


def _deep_job_v6(spec, excess, level, D_blocks, store_codes, methods):
    rng = np.random.default_rng(spec["seed"])
    k, lits, d = spec["k"], spec["lits"], spec["lits"]
    A_ = binary_descriptors(rng, k)
    attrs = rng.choice(k, lits, replace=False)
    planted_desc = tuple(sorted((int(j), 0) for j in attrs))
    planted = mask_of(A_, planted_desc)
    rows = np.flatnonzero(planted)
    keep = rows[rng.random(len(rows)) >= spec["outliers"]]
    exc = excess.copy()
    exc[keep] += spec["delta"] * POST[None, :] * level[keep, None]
    rel = np.clip(exc / level[:, None], -3, 3)

    t = time.time()
    Db = update_rows(D_blocks, block_means(rel), keep)
    comps = snn_components(Db, max_size=int(MAX_COV * len(level)))
    overall = A_.mean(axis=0)
    comp_conds = [enriched_conditions(mc, A_, overall) for mc in comps]
    seed_time = time.time() - t
    cands = set()
    for conds in comp_conds:
        for size in range(1, min(d, len(conds)) + 1):
            cands.update(tuple(sorted(c)) for c in itertools.combinations(conds, size))
    t = time.time()
    support = itemset_support(comp_conds, d)
    frequent = sorted((s for s, n in support.items() if n >= MIN_SUPPORT), key=lambda s: (-support[s], -len(s)))
    itemset_time = time.time() - t
    planted_in = [planted[list(mc)].mean() for mc in comps]
    seed_row = {"components": len(comps), "mean_comp_size": np.mean([len(c) for c in comps]) if comps else 0,
                "best_comp_purity": max(planted_in, default=0.0),
                "comps_mostly_planted": int(sum(p >= 0.5 for p in planted_in)),
                "planted_conds_in_top": max((len(set(cc) & set(planted_desc)) for cc in comp_conds), default=0),
                "planted_support": support.get(planted_desc, 0), "frequent_itemsets": len(frequent),
                "candidates": len(cands), "seed_time_s": seed_time}

    out = []
    for method in methods:
        phi = MonetaryQuality(exc, level, store_codes, relative=True, min_cov=MIN_COV_V5)
        phi.track, phi.trace = planted, []
        stats = Counter()
        t0 = time.time()
        offset = 0.0
        if method.startswith("BS"):
            top, stats = beam_search(A_, phi, w=int(method.split("w=")[1]), d=d, prune=True)
        elif method.startswith("BUS→BS pool"):
            offset = seed_time
            votes = Counter(c for conds in comp_conds for c in conds)
            top, stats = beam_search(A_, phi, w=10, d=d, prune=True, pool=[c for c, _ in votes.most_common(POOL_SIZE)])
        elif method.startswith("BUS itemsets"):
            offset = seed_time + itemset_time
            top, stats = evaluate_candidates(A_, phi, set(frequent), prune=True)
            if method.endswith("+guided beam"):
                rs = ResultSet(Q)
                for q_, d_, m_ in top:
                    rs.insert(q_, d_, m_)
                top = guided_beam(A_, phi, top, rs, stats, d, support)
        else:
            offset = seed_time
            top, stats = evaluate_candidates(A_, phi, cands, prune=True)
            if method.endswith("+local"):
                rs = ResultSet(Q)
                for q_, d_, m_ in top:
                    rs.insert(q_, d_, m_)
                top = local_search_d(A_, phi, top, rs, stats, d)
        trace = [(tt - t0 + offset, q_, jac, valid) for tt, q_, jac, valid in phi.trace]
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = dict(spec)
        row.update({"method": method, "planted_n": int(planted.sum()), "conditions": 2 * k,
                    "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
                    "rank": ranks[0] if ranks else np.nan, "evaluated": len(trace),
                    "runtime_s": time.time() - t0 + offset, "local_evaluated": stats.get("local_evaluated", 0),
                    **seed_row})
        for b in BUDGETS + [6400, 12800]:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s in SECONDS:
            row[f"rank_at_{s}s"] = _rank_of_planted([e for e in trace if e[0] <= s])
        out.append(row)
    return out

In [ ]:
# ---- calibration (identical to v5, so the task list is reproducible across restarts)
rng = np.random.default_rng(SEED + 5)
SIGMA_LITS = {}
for lits in LITS_V5:
    rates = []
    for _ in range(200):
        A_ = binary_descriptors(rng, 20)
        mask = mask_of(A_, tuple((int(j), 0) for j in rng.choice(20, lits, replace=False)))
        rates.append(excess[mask][:, POST].sum() / (level[mask].sum() * POST.sum()))
    SIGMA_LITS[lits] = float(np.std(rates))
print("null spread per description length:", {k: round(v, 4) for k, v in SIGMA_LITS.items()})

D_blocks = pairwise_cityblock(block_means(rel_dev))
specs = []
for k in K_ATTRS:
    for lits in LITS_V5:
        for snr in SNR_V5:
            for outliers in OUTLIERS_V5:
                for rep in range(REPS_V5):
                    specs.append({"dataset": DATASET, "k": k, "lits": lits, "snr": snr, "outliers": outliers, "rep": rep,
                                  "delta": snr * SIGMA_LITS[lits], "seed": int(rng.integers(1 << 31))})

# ---- resumable, batched execution: each batch is appended to the CSV; progress.txt shows how far it is
out_csv = f"{RESULTS_DIR}/{DATASET}_deep.csv"
progress = f"{RESULTS_DIR}/progress.txt"
done = set(pd.read_csv(out_csv)["seed"]) if os.path.exists(out_csv) else set()
todo = [s for s in specs if s["seed"] not in done]
# interleave cheap and expensive tasks so every batch takes a similar time
todo = sorted(todo, key=lambda s: (s["rep"], s["lits"], s["k"]))
print(f"{len(specs)} tasks x {len(METHODS_V5)} methods; {len(done)} already done, {len(todo)} to run")
start = time.time()
for b in range(0, len(todo), BATCH):
    batch = todo[b:b + BATCH]
    parts = Parallel(n_jobs=N_JOBS)(delayed(_deep_job_v6)(s, excess, level, D_blocks, store_codes, METHODS_V5) for s in batch)
    rows = pd.DataFrame([r for p in parts for r in p])
    rows.to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
    finished = len(done) + b + len(batch)
    elapsed = time.time() - start
    eta = elapsed / (b + len(batch)) * (len(todo) - b - len(batch))
    with open(progress, "w") as f:
        f.write(f"{finished} / {len(specs)} tasks done, {elapsed / 60:.1f} min elapsed, ~{eta / 60:.1f} min left\n")
deep = pd.read_csv(out_csv)
print(f"done in {time.time() - start:.0f}s")

In [ ]:
from scipy.stats import binomtest

deep["found"] = deep["rank"] <= Q
print("share of true subgroups in the top-20")
display(deep.pivot_table(index=["k", "lits"], columns="method", values="found")[METHODS_V5].round(2))
display(deep.groupby("method")[["found", "evaluated_hit", "evaluated", "runtime_s"]].mean().reindex(METHODS_V5).round(3))

key = ["k", "lits", "snr", "outliers", "rep"]
w = deep.pivot_table(index=key, columns="method", values="found").astype(bool)
rows = []
for group, sub in [("all", w)] + [(f"k={k}, lits={l}", w.xs((k, l), level=("k", "lits"), drop_level=False))
                                   for k in K_ATTRS for l in LITS_V5]:
    for a in METHODS_V5[2:] + ["BS w=50"]:
        wins = int((sub[a] & ~sub["BS w=10"]).sum())
        losses = int((~sub[a] & sub["BS w=10"]).sum())
        rows.append({"group": group, "method": a, "found": sub[a].mean(), "BS w=10": sub["BS w=10"].mean(),
                     "wins": wins, "losses": losses, "p": binomtest(wins, wins + losses).pvalue if wins + losses else 1.0})
paired = pd.DataFrame(rows)
paired.to_csv(f"{RESULTS_DIR}/{DATASET}_deep_paired.csv", index=False)
display(paired.round(3))
print("seed diagnostics")
display(deep.drop_duplicates(key).groupby(["k", "lits"])[["components", "mean_comp_size", "best_comp_purity",
                                                          "comps_mostly_planted", "planted_conds_in_top", "candidates",
                                                          "seed_time_s"]].mean().round(2))